<h1>Structured Output</h1>The response of the model can be parsed into structured output(according to the requirement and schema)

This can be done using 
1) Pydantic
2) TypedDict

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY']=os.getenv('GOOGLE_API_KEY')


In [9]:
from langchain.chat_models import init_chat_model
model=init_chat_model(model='groq:qwen/qwen3-32b')
model1=init_chat_model(model='groq:qwen/qwen3-32b')
model2=init_chat_model(model='groq:qwen/qwen3-32b')

In [9]:
for chunks in model.stream("give the details of the movie inception"):
    print(chunks.content, end="", flush=True)

<think>
Okay, so I need to figure out all the details about the movie Inception. Let me start by recalling what I know. It's a 2010 movie directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio, and he plays Dom Cobb. The title is Inception, which I think has something to do with entering someone's dreams. 

The plot is a bit complex. I remember that the concept involves entering people's dreams to implant or extract ideas. There's this device called the totem, which is used to determine if someone is in a dream or not. Each character has a different totem. For example, Cobb has a spinning top. The idea is that if the top keeps spinning, he's in a dream, and if it falls, he's back in reality. 

The main team consists of Cobb, Arthur (played by Joseph Gordon-Levitt), Ariadne (Elliot Page), Eames (Tom Hardy), and Yusuf (Dileep Rao). There's also a character named Mal (Marion Cotillard), Cobb's wife, who appears in his dreams. She's a projection of his guilt from her dea

In [20]:
print(model)
print(type(model))

first=_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026CA406B950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026CA40BF230>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie released', 'type': 'integer'}, 'cast': {'description': 'Includes all the actors of the movie', 'items': {'properties': {'name': {'description': 'name o

<h3>Structuring using Pydantic</h3>
Pydantic allows us to structure output and it also has <b>schema validation</b>, descriptions and <b>nested  structures</b>  

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="The year the movie released")
    genre:list[str]=Field(description="Include all the genres the movie belongs to")

model_with_structure=model.with_structured_output(Movie) 
response=model_with_structure.invoke("Give the details of the movie  Jersey")
response

Movie(title='Jersey', year=2019, genre=['Sports Drama', 'Biographical'])

<h4>Pydantic with nested structures</h4>

<h4 style="color:red">Error: if we rerun the statement "model.with_structured_output(MovieDetails)"<h3>
we are creating runnable sequence, when we again assign it to the same variable model, model is getting overwritten with runnable sequence( From groq model to Runnable Sequence).
if we re run the statement, we are again and again doing the error

In [4]:
class Actor(BaseModel):
    name:str=Field(description="name of the actor")
    role:str=Field(description="What role did the actor play in that movie")

class MovieDetails(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="The year the movie released")
    cast:list[Actor]=Field(description="Includes all the actors of the movie")
    genre:list[str]=Field(description="Include all the genres the movie belongs to")

model=model.with_structured_output(MovieDetails) 
response=model.invoke("Give the details of the movie Nani's Jersey")
response

MovieDetails(title="Nani's Jersey", year=2023, cast=[Actor(name='Nani', role='Jersey'), Actor(name='Sai Pallavi', role='Supporting Role')], genre=['Drama', 'Sports'])

<h3>TypedDict</h3>
This is just an alternative to inbuilt python inbuilt typing, it is used when we need no run time validation

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDetails(TypedDict):
    title:Annotated[str,"Title of the movie"]
    year:Annotated[int,"The year the movie released"]
    cast:Annotated[list[str],"Includes all the actors of the movie"]
    genre:Annotated[list[str],"Include all the genres the movie belongs to"]

modelling=model2.with_structured_output(MovieDetails)

<h4 style="color:cyan">There is another way to tell model to give structured response, i.e, directly while creating the agent ,giving it the response format
<h2>Using DataClass</h2>

In [28]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class Movie1(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="The year the movie released")
    genre:list[str]=Field(description="Include all the genres the movie belongs to")

model3=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=Movie1
)

# generally without dictionary it works since we kept response format 
response=model3.invoke({'messages':["Tell the details of tenet movie"]})


In [29]:
response

{'messages': [HumanMessage(content='Tell the details of tenet movie', additional_kwargs={}, response_metadata={}, id='6223a6ba-18a1-4d38-879f-577696ad360b'),
  AIMessage(content='{\n"title": "Tenet",\n"year": 2020,\n"genre": ["Action", "Science Fiction", "Thriller", "Spy"]\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7f7a-5ce2-7f31-ab0a-7b8cd9ab1f5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 38, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}})],
 'structured_response': Movie1(title='Tenet', year=2020, genre=['Action', 'Science Fiction', 'Thriller', 'Spy'])}